# Day 42 — Counting the Clues

## Naive Bayes

**Boston University · Data Science Program**

---

The slides carry the ideas. This notebook is where you check them — recount the
messages, break the model on purpose, and watch it get repaired.

Everything today comes from one sentence:

> **Conditioning on evidence means throwing away the rows that do not match, and
> counting what is left.**

| | |
|---|---|
| 1 | Count twenty messages by hand, then get the same answer from Bayes' rule |
| 2 | Watch the honest calculation run out of data at two clues |
| 3 | Measure exactly how wrong the "naive" assumption is |
| 4 | Break the model with a single zero, then repair it |
| 5 | Build the whole classifier yourself in fifteen lines |
| 6 | Meet the three sklearn flavours, and see what it costs |

In [ ]:
import math, time, warnings
import numpy as np
warnings.simplefilter("ignore")

from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(5, shuffle=True, random_state=0)
print("ready")

---
# 1 · Twenty messages

Here is the entire dataset. Twenty short messages, already labelled. Small enough
that you can check every number in this notebook by eye.

In [ ]:
SPAM = [
    "free money click now",
    "win a free prize",
    "claim your free gift now",
    "free entry win cash",
    "urgent claim your prize",
    "cheap pills buy now",
    "you win free credit",
    "limited free offer buy now",
]
HAM = [
    "meeting at ten tomorrow",
    "can we move the meeting",
    "lunch after the meeting",
    "see you at the lab",
    "the report is ready",
    "please review my code",
    "meeting notes attached",
    "are you free for a meeting",
    "class starts at nine",
    "submit the assignment today",
    "meeting room changed",
    "thanks for the help",
]

MSGS = [(m, 1) for m in SPAM] + [(m, 0) for m in HAM]     # 1 = spam, 0 = ham

def has(msg, word):
    return word in msg.split()

def count(word, label=None):
    """How many messages contain `word`? Optionally only those with this label."""
    return sum(1 for m, y in MSGS if has(m, word) and (label is None or y == label))

def n(label=None):
    return sum(1 for _, y in MSGS if label is None or y == label)

print("messages:", n(), " spam:", n(1), " ham:", n(0))
print()
for w in ["free", "meeting"]:
    print(f'"{w}":  spam {count(w,1)}/{n(1)}    ham {count(w,0)}/{n(0)}    '
          f'total {count(w)}/{n()}')

## 1.1 · The question, answered by covering up rows

A new message arrives containing the word **free**. How likely is it to be spam?

Forget formulas. Just ignore every message that does not contain "free" — they say
nothing about this one — and count what survives.

In [ ]:
survivors = [(m, y) for m, y in MSGS if has(m, "free")]

print(f"{len(survivors)} messages contain 'free':\n")
for m, y in survivors:
    print(f"   {'SPAM' if y else 'ham '}   {m}")

spam_left = sum(y for _, y in survivors)
print(f"\nof those {len(survivors)}, {spam_left} are spam")
print(f"P(spam | 'free') = {spam_left}/{len(survivors)} = {spam_left/len(survivors):.4f}")

That is the whole idea, and you did it without a single formula.

## 1.2 · The same answer from Bayes' rule

Bayes' rule is not a new fact about the world. It is a way of reaching that same
count when the table is too big to cover up with your hand.

$$P(\text{class} \mid \text{clue}) \;=\;
\frac{P(\text{clue} \mid \text{class}) \times P(\text{class})}{P(\text{clue})}$$

In [ ]:
P_spam = n(1) / n()                      # the prior      8/20
P_free_given_spam = count("free", 1) / n(1)   # the likelihood 6/8
P_free = count("free") / n()             # the evidence   7/20

posterior = P_free_given_spam * P_spam / P_free

print(f"  prior       P(spam)        = {n(1)}/{n()}  = {P_spam:.4f}")
print(f"  likelihood  P(free | spam) = {count('free',1)}/{n(1)}   = {P_free_given_spam:.4f}")
print(f"  evidence    P(free)        = {count('free')}/{n()}  = {P_free:.4f}")
print()
print(f"  posterior   P(spam | free) = {P_free_given_spam:.4f} x {P_spam:.4f}"
      f" / {P_free:.4f} = {posterior:.4f}")
print()
print("  by covering up rows we got :", round(spam_left / len(survivors), 4))
print("  they agree                 :", abs(posterior - spam_left/len(survivors)) < 1e-12)

## 1.3 · The trap: these two are not the same number

`P(spam | free)` and `P(free | spam)` look alike and mean completely different
things. Swapping them is the most common mistake in all of probability — it is
known as the **prosecutor's fallacy**.

In [ ]:
print(f"P(spam | free)  'it says free, so how likely is spam?'   "
      f"{count('free',1)}/{count('free')} = {count('free',1)/count('free'):.4f}")
print(f"P(free | spam)  'it is spam, so how likely to say free?'  "
      f"{count('free',1)}/{n(1)} = {count('free',1)/n(1):.4f}")

---
# 2 · Two clues, and the data runs out

One clue was easy. Now the message says **"free meeting"** — one word pointing at
spam, one pointing at ham.

To do this *honestly* we would need `P(free AND meeting | class)`: the number of
messages containing both. Let's see how much data is left to count.

In [ ]:
def joint(w1, w2, label):
    return sum(1 for m, y in MSGS if y == label and has(m, w1) and has(m, w2))

print("ham messages, split four ways:")
print(f"   free YES, meeting YES :  {joint('free','meeting',0)}")
print(f"   free YES, meeting NO  :  {count('free',0) - joint('free','meeting',0)}")
print(f"   free NO,  meeting YES :  {count('meeting',0) - joint('free','meeting',0)}")
print(f"   free NO,  meeting NO  :  "
      f"{n(0) - count('free',0) - count('meeting',0) + joint('free','meeting',0)}")
print(f"   total                 :  {n(0)}")

Two words already split twelve messages into four boxes, and the box we actually
care about holds **one message**. Estimating a probability from one message is not
estimating anything.

And it gets worse fast — each extra word *doubles* the number of boxes:

In [ ]:
print(f"{'words':>7}  {'boxes to fill':>20}")
for k in [1, 2, 3, 10, 20]:
    print(f"{k:>7}  {2**k:>20,}")
print(f"{1000:>7}  {'about 1.07e301':>20}   (a 302-digit number)")
print()
print("...and we have 20 messages.")

This is not a problem you fix with a bigger computer or more data. You fix it by
refusing to ask the question.

## 2.1 · The naive shortcut

Stop counting combinations. Count each word **on its own**, then multiply:

$$P(\text{free and meeting} \mid \text{ham}) \;\approx\;
P(\text{free} \mid \text{ham}) \times P(\text{meeting} \mid \text{ham})$$

Note the $\approx$. This is not a true statement — it is a decision to pretend.
Let's measure exactly how false it is.

In [ ]:
exact = joint("free", "meeting", 0) / n(0)
naive = (count("free", 0) / n(0)) * (count("meeting", 0) / n(0))

print(f"  EXACT  P(free and meeting | ham) = {joint('free','meeting',0)}/{n(0)}"
      f"        = {exact:.4f}")
print(f"  NAIVE  P(free|ham) x P(meeting|ham) = "
      f"{count('free',0)}/{n(0)} x {count('meeting',0)}/{n(0)} = {naive:.4f}")
print()
print(f"  the naive answer is wrong by a factor of {exact/naive:.2f}")
print()
print("  and here is the message that breaks it:")
for m, y in MSGS:
    if y == 0 and has(m, "free") and has(m, "meeting"):
        print("     ", repr(m))

**The assumption is false and we just proved it on twenty messages.** "Free" and
"meeting" travel together in real sentences — *"are you free for a meeting"* — so
treating them as unrelated under-counts that corner by half.

### So why use it?

Because of what we ask the model for at the end. To **label** a message we only need
to know which of the two totals is bigger — we never use the probability itself.
The naive assumption distorts both totals, usually in the same direction, so the
**order** tends to survive even when the numbers do not.

That makes naive Bayes a good classifier and a poor estimator of probability. Trust
`predict()`. Be careful quoting `predict_proba()`.

---
# 3 · One zero ruins everything

Now actually classify **"free meeting"**. Multiply the prior by one likelihood per
word, for each class, and see which total is bigger.

In [ ]:
def score(words, label, alpha=0.0):
    """prior x product of per-word likelihoods, with Laplace smoothing alpha."""
    p = n(label) / n()
    for w in words:
        p *= (count(w, label) + alpha) / (n(label) + 2 * alpha)
    return p

words = ["free", "meeting"]
s_spam = score(words, 1)
s_ham  = score(words, 0)

print(f"  P(meeting | spam) = {count('meeting',1)}/{n(1)} = "
      f"{count('meeting',1)/n(1):.4f}   <-- nobody said 'meeting' in any spam")
print()
print(f"  spam score = {s_spam:.6f}")
print(f"  ham  score = {s_ham:.6f}")
print(f"  P(spam | 'free meeting') = {s_spam/(s_spam+s_ham):.4f}")

**Exactly zero.** Not "unlikely" — impossible. One word that happens never to appear
in our eight spam messages has overruled every other piece of evidence, and it would
keep overruling it if the message were a thousand words of obvious spam.

That is not confidence. It is a gap in the data being mistaken for knowledge.

## 3.1 · Laplace smoothing

The fix is almost embarrassingly simple: **add a small amount to every count** before
dividing, so nothing is ever exactly zero.

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{n_c + 2\alpha}$$

In [ ]:
print(f"{'alpha':>7}  {'P(meeting|spam)':>17}  {'spam score':>12}  {'ham score':>11}  "
      f"{'P(spam)':>9}  verdict")
for a in [0.0, 0.01, 0.1, 0.5, 1.0, 5.0]:
    ss, sh = score(words, 1, a), score(words, 0, a)
    pm = (count("meeting", 1) + a) / (n(1) + 2 * a)
    print(f"{a:>7}  {pm:>17.4f}  {ss:>12.6f}  {sh:>11.6f}  "
          f"{ss/(ss+sh):>9.4f}  {'spam' if ss > sh else 'ham'}")

At `alpha = 1` the answer is **39.5% spam** — still ham, but now for a reason that
survives one missing word rather than being decided by it.

Note what smoothing did *not* do: it did not change the verdict. "Meeting" really is
strong evidence for ham here. It only stopped the model being *certain* about it.

`alpha` is sklearn's only real knob for naive Bayes, and it defaults to 1.

## 3.2 · Why the real code adds logs

Multiply enough small numbers together and the answer stops existing.

In [ ]:
for k in [100, 150, 200]:
    product = 0.01 ** k
    logsum  = k * math.log(0.01)
    print(f"  {k:>3} clues at p=0.01 ->  product {product:>10.3e}"
          f"   log-sum {logsum:>10.3f}   underflowed: {product == 0.0}")

print()
print("  smallest positive number a float64 can hold:", np.nextafter(0, 1))

At 200 clues the product is **exactly 0.0** — smaller than the smallest number the
computer can store, so it rounds to nothing and takes the answer with it. A real spam
filter looks at far more than 200 words.

Logs fix this. They turn multiplication into addition, and they never change the
order: if one total is bigger, so is its log. Since all we need is *which class wins*,
nothing is lost.

---
# 4 · Build the whole classifier yourself

You now know every piece. Here is the entire model in fifteen lines — fit by counting,
predict by adding logs.

In [ ]:
class MyNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, docs, labels):
        self.classes = sorted(set(labels))
        self.vocab = sorted({w for d in docs for w in d.split()})
        self.log_prior, self.log_lik = {}, {}
        for c in self.classes:
            in_c = [d for d, y in zip(docs, labels) if y == c]
            self.log_prior[c] = math.log(len(in_c) / len(docs))
            self.log_lik[c] = {
                w: math.log((sum(1 for d in in_c if w in d.split()) + self.alpha)
                            / (len(in_c) + 2 * self.alpha))
                for w in self.vocab}
        return self

    def _total(self, doc, c):
        t = self.log_prior[c]
        for w in set(doc.split()):
            if w in self.vocab:                 # ignore words never seen in training
                t += self.log_lik[c][w]
        return t

    def predict(self, doc):
        return max(self.classes, key=lambda c: self._total(doc, c))


docs   = [m for m, _ in MSGS]
labels = [y for _, y in MSGS]
model = MyNaiveBayes(alpha=1.0).fit(docs, labels)

tests = ["free money now", "are you free tomorrow", "meeting at the lab",
         "free meeting", "win cash prize", "please review the report"]
for d in tests:
    print(f"  {'SPAM' if model.predict(d) else 'ham ' }   {d}")

Fifteen lines, no optimiser, no gradient, no iterations. It fits in a single pass of
counting — which is exactly why it is the fastest model in this course.

> Look at `_total`: it **adds** the log-likelihoods rather than multiplying
> probabilities, exactly as section 3.2 said it must.

**Look at the second prediction.** *"are you free tomorrow"* is plainly a friendly
message, and the model calls it spam. It is not broken — it has seen the word "free"
in six of eight spam messages and in only one ham message, and with twenty messages of
training data that one word swamps everything else.

This is worth sitting with. The model is behaving exactly as designed; the *data* is
too small. Give it ten thousand messages and "free" stops being decisive, because
plenty of ordinary mail would contain it. Most surprising model behaviour is like this
— correct arithmetic over insufficient evidence.

---
# 5 · The three flavours in sklearn

They differ in one place only: how the likelihood of a single clue is worked out.

| flavour | use when a column is | typical job |
|---|---|---|
| `MultinomialNB` | a **count** (how often a word appears) | text, documents, spam |
| `GaussianNB` | a **measurement** (height, price, a reading) | medical and scientific tables |
| `BernoulliNB` | a **yes/no** (the word is there, or not) | short text, tick-box features |

## 5.1 · GaussianNB: what to do with a measurement

Counting works for words, which are present or absent. It cannot work for `0.0413`.
So for each class, fit a bell curve to that column using only its own mean and spread,
and read the height of each curve at the new value.

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)
col = 7                                    # 'mean concave points'
f = Xb[:, col]

for lab, name in [(0, "malignant"), (1, "benign   ")]:
    print(f"  {name}  mean {f[yb==lab].mean():.4f}   sd {f[yb==lab].std():.4f}")

def bell(x, m, s):
    return math.exp(-0.5 * ((x - m) / s) ** 2) / (s * math.sqrt(2 * math.pi))

m0, s0 = f[yb == 0].mean(), f[yb == 0].std()
m1, s1 = f[yb == 1].mean(), f[yb == 1].std()

print()
for x in [0.02, 0.05, 0.12]:
    h0, h1 = bell(x, m0, s0), bell(x, m1, s1)
    print(f"  a patient reading {x:.2f}:  malignant curve {h0:7.3f}"
          f"   benign curve {h1:7.3f}   -> {'malignant' if h0 > h1 else 'benign'}")

Whichever curve is taller wins (after multiplying by the prior). That is the only
change — Bayes' rule, the naive assumption and the logs are all exactly as before.

## 5.2 · Does it need scaling?

k-NN and SVMs collapsed without `StandardScaler`. Naive Bayes never measures a
distance between two rows, so stretching a column should not matter at all.

In [ ]:
raw    = cross_val_score(GaussianNB(), Xb, yb, cv=CV).mean()
scaled = cross_val_score(make_pipeline(StandardScaler(), GaussianNB()),
                         Xb, yb, cv=CV).mean()
print(f"  GaussianNB raw     {raw:.4f}")
print(f"  GaussianNB scaled  {scaled:.4f}")
print()
# ...and with sklearn's tiny variance floor switched off, they are bit-identical
raw0    = cross_val_score(GaussianNB(var_smoothing=0), Xb, yb, cv=CV).mean()
scaled0 = cross_val_score(make_pipeline(StandardScaler(), GaussianNB(var_smoothing=0)),
                          Xb, yb, cv=CV).mean()
print(f"  with var_smoothing=0:  raw {raw0:.4f}   scaled {scaled0:.4f}"
      f"   identical: {abs(raw0-scaled0) < 1e-12}")

The two differ by less than a point, and the second pair shows why: with sklearn's
tiny variance floor (`var_smoothing`, on by default) switched off, scaled and unscaled
give **exactly** the same number.

So the rule really does hold — **naive Bayes does not need scaling** — and the small
wobble at the default setting is a numerical detail, not the scaling.

---
# 6 · What it costs

Time to be honest about where naive Bayes sits. Handwritten digits: 1,797 rows,
64 columns of pixel counts, ten classes.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
print("rows", Xd.shape[0], " columns", Xd.shape[1], " classes", len(set(yd)))
print("baseline (most common digit):", round(np.bincount(yd).max() / len(yd), 4))
print()

models = [
    ("MultinomialNB", MultinomialNB()),
    ("BernoulliNB", BernoulliNB()),
    ("GaussianNB", GaussianNB()),
    ("k-NN (k=5)", make_pipeline(StandardScaler(), KNeighborsClassifier())),
    ("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
    ("SVM (rbf)", make_pipeline(StandardScaler(), SVC())),
    ("random forest 200", RandomForestClassifier(n_estimators=200, random_state=0)),
]

def best_of_3(mo):
    """warm up, then take the fastest of three runs -- single runs are noisy"""
    cross_val_score(mo, Xd, yd, cv=CV)
    times = []
    for _ in range(3):
        t = time.perf_counter()
        cross_val_score(mo, Xd, yd, cv=CV)
        times.append(time.perf_counter() - t)
    return min(times)

print(f"{'model':<22}{'accuracy':>10}{'seconds':>12}")
timing = {}
for name, mo in models:
    acc = cross_val_score(mo, Xd, yd, cv=CV).mean()
    timing[name] = best_of_3(mo)
    print(f"{name:<22}{acc:>10.4f}{timing[name]:>12.3f}")

print()
print(f"random forest is ~{timing['random forest 200']/timing['MultinomialNB']:.0f}x "
      f"slower than MultinomialNB")
print("(absolute times depend on the machine, and the NB numbers are a few")
print(" milliseconds, so expect this ratio to wobble between runs -- the point")
print(" is the order of magnitude, not the exact figure)")

Read **both** columns. Naive Bayes comes last on accuracy — about eight points behind
the SVM — and it is roughly two hundred times faster than the forest.

That is the trade, and it is the honest reason to reach for it: not because it wins,
but because it answers in ten milliseconds and tells you what the floor looks like
before you spend an afternoon on something better.

---
# 7 · Summary

| | |
|---|---|
| **The goal** | `P(class \| evidence)` — what we want but cannot count directly. |
| **Conditioning** | Throw away the rows that do not match the evidence, then count. |
| **Bayes' rule** | likelihood × prior ÷ evidence. Turns the count we *can* make into the answer we want. |
| **Naive** | Pretend the clues are unrelated, so you multiply instead of counting combinations. |
| **It is false** | Provably — off by a factor of 2 on our twenty messages. The ranking survives anyway. |
| **Smoothing** | Add `alpha` to every count, so one unseen word cannot zero the product. |
| **Logs** | Add logs instead of multiplying, or 200 clues underflow to exactly zero. |
| **Flavours** | Counts → Multinomial. Measurements → Gaussian. Yes/no → Bernoulli. |
| **No scaling** | It never measures distance, so scaling cannot change the answer. |

### Next: Day 43

Naive Bayes needed labels — someone had to mark those twenty messages as spam or ham.
Next we drop the labels entirely and ask whether a model can find the groups on its own.